In [1]:
!pip install tree-sitter tree-sitter-python chromadb openai tqdm rank_bm25 langgraph


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 667.5/667.5 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 75.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 57.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [2]:
import os
import sys
import uuid
import chromadb
import tree_sitter
import tree_sitter_python
from openai import OpenAI
from rank_bm25 import BM25Okapi
from typing import TypedDict, List, Tuple, Optional
from langgraph.graph import StateGraph, END


In [3]:
def chunk_repo(repo_path):
    """
    Recursively scans the directory specified by repo_path for Python (.py) files,
    parses each file using Tree-sitter, and extracts top-level function definitions,
    class methods, and non-trivial class-level statements (like docstrings and properties)
    as separate chunks.
    
    Parameters:
        repo_path (str): The absolute or relative file path to the repository directory we want to scan.
        
    Returns:
        list of dict: A list of dictionaries, where each dictionary represents an extracted code chunk.
    """
    chunks = []
    
    # Conceptually, the tree-sitter-python library provides the grammar rules for the Python language,
    # and the tree-sitter core library uses those rules to initialize a Parser. The Parser reads raw code
    # and creates a structured tree representing the code's grammar (functions, classes, variables, etc.).
    try:
        language = tree_sitter.Language(tree_sitter_python.language())
        parser = tree_sitter.Parser(language)
    except Exception as error:
        print("Error initializing Tree-sitter parser:", error)
        return chunks

    # Walk the directory structure recursively to find Python source files
    for root_dir, dirs, files in os.walk(repo_path):
        for file in files:
            if file.endswith('.py'):
                full_path = os.path.join(root_dir, file)
                # Compute relative path using forward slashes for cross-platform consistency
                rel_path = os.path.relpath(full_path, repo_path).replace('\\', '/')
                
                try:
                    # Read python file contents with utf-8 encoding and replace invalid characters
                    with open(full_path, 'r', encoding='utf-8', errors='replace') as python_file:
                        content = python_file.read()
                except Exception as error:
                    print("Warning: Failed to read file", rel_path, ":", error)
                    continue

                try:
                    content_bytes = content.encode('utf-8')
                    # Parse the bytes of the file content to build the syntax tree
                    tree = parser.parse(content_bytes)
                    
                    if tree.root_node.has_error:
                        print("Warning: Syntax errors in file", rel_path, ". Skipping.")
                        continue
                    
                    # Extract top-level function and class definitions
                    for child in tree.root_node.children:
                        # 1. Handle top-level functions
                        if child.type == 'function_definition':
                            name_node = child.child_by_field_name('name')
                            name = name_node.text.decode('utf-8', errors='replace') if name_node else 'unknown'
                            
                            code_text = content_bytes[child.start_byte:child.end_byte].decode('utf-8', errors='replace')
                            start_line = child.start_point[0] + 1
                            end_line = child.end_point[0] + 1
                            
                            chunks.append({
                                "code": code_text,
                                "file_path": rel_path,
                                "class_name": "",
                                "name": name,
                                "type": "function",
                                "start_line": start_line,
                                "end_line": end_line
                            })
                        
                        # 2. Handle class definitions (split into methods and class-level properties)
                        elif child.type == 'class_definition':
                            class_name_node = child.child_by_field_name('name')
                            class_name = class_name_node.text.decode('utf-8', errors='replace') if class_name_node else 'unknown'
                            
                            # Find the class body block
                            class_block = None
                            for c in child.children:
                                if c.type == 'block':
                                    class_block = c
                                    break
                            
                            if class_block is not None:
                                for member in class_block.children:
                                    if member.type == 'function_definition':
                                        # Extract method inside class
                                        method_name_node = member.child_by_field_name('name')
                                        method_name = method_name_node.text.decode('utf-8', errors='replace') if method_name_node else 'unknown'
                                        
                                        method_code = content_bytes[member.start_byte:member.end_byte].decode('utf-8', errors='replace')
                                        m_start = member.start_point[0] + 1
                                        m_end = member.end_point[0] + 1
                                        
                                        chunks.append({
                                            "code": method_code,
                                            "file_path": rel_path,
                                            "class_name": class_name,
                                            "name": method_name,
                                            "type": "method",
                                            "start_line": m_start,
                                            "end_line": m_end
                                        })
                                    else:
                                        # Extract class-level code (docstrings, properties)
                                        member_text = content_bytes[member.start_byte:member.end_byte].decode('utf-8', errors='replace').strip()
                                        # Skip trivial items (empty, comments only, single semicolons, passes)
                                        if member_text and member_text not in (';', 'pass') and len(member_text) > 5:
                                            m_start = member.start_point[0] + 1
                                            m_end = member.end_point[0] + 1
                                            
                                            chunks.append({
                                                "code": member_text,
                                                "file_path": rel_path,
                                                "class_name": class_name,
                                                "name": class_name + "_definition",
                                                "type": "class_definition",
                                                "start_line": m_start,
                                                "end_line": m_end
                                            })
                except Exception as error:
                    print("Warning: Failed to parse file", rel_path, ":", error)
                    continue
                    
    return chunks


In [4]:
def embed_and_store(chunks, collection_name, persist_dir, nvidia_api_key):
    """
    Converts code chunks into mathematical embeddings using NVIDIA NIM API (llama-3.2-nv-embedqa-1b-v2)
    and stores those embeddings along with their metadata inside a local, persistent Chroma vector database.
    """
    if not chunks:
        print("No chunks to embed and store.")
        return

    # Conceptually, we connect to the NVIDIA NIM API to generate embeddings in the cloud.
    # This prevents downloading heavy model weights (like PyTorch/Transformers) to your laptop.
    print("Initializing NVIDIA NIM API connection...")
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=nvidia_api_key
    )

    # Conceptually, chromadb is a database that stores these high-dimensional vectors and builds a search index
    # (using HNSW - Hierarchical Navigable Small World).
    print("Connecting to persistent Chroma database at:", persist_dir)
    client_db = chromadb.PersistentClient(path=persist_dir)

    # Get or create collection with cosine similarity configuration
    collection = client_db.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )       

    total_chunks = len(chunks)
    batch_size = 50

    from tqdm import tqdm

    print("Starting embedding and storage of", total_chunks, "chunks using NVIDIA NIM...")
    for i in tqdm(range(0, total_chunks, batch_size), desc="Embedding and storing chunks"):
        batch = chunks[i:i + batch_size]
        
        batch_codes = []
        batch_texts_to_embed = []
        for chunk in batch:
            batch_codes.append(chunk['code'])
            # Create a structured prefix text that includes the class name for method chunks
            if chunk['type'] == 'method':
                text_to_embed = "File: " + chunk['file_path'] + "\nClass: " + chunk['class_name'] + "\nMethod: " + chunk['name'] + "\nType: method\n\n" + chunk['code']
            elif chunk['type'] == 'class_definition':
                text_to_embed = "File: " + chunk['file_path'] + "\nClass: " + chunk['class_name'] + "\nType: class_definition\n\n" + chunk['code']
            else:
                text_to_embed = "File: " + chunk['file_path'] + "\nName: " + chunk['name'] + "\nType: function\n\n" + chunk['code']
            batch_texts_to_embed.append(text_to_embed)

        # Call NVIDIA NIM API to generate embeddings for the batch
        response = client.embeddings.create(
            input=batch_texts_to_embed,
            model="nvidia/llama-nemotron-embed-1b-v2",
            extra_body={"input_type": "passage", "truncate": "NONE"}
        )
        
        # Extract embeddings list from response
        batch_embeddings = []
        for data in response.data:
            batch_embeddings.append(data.embedding)

        # Generate unique IDs and extract metadata
        batch_ids = []
        for _ in batch:
            batch_ids.append(uuid.uuid4().hex)
            
        batch_metadatas = []
        for chunk in batch:
            metadata_dict = {
                "file_path": chunk["file_path"],
                "class_name": chunk["class_name"],
                "name": chunk["name"],
                "type": chunk["type"],
                "start_line": chunk["start_line"],
                "end_line": chunk["end_line"]
            }
            batch_metadatas.append(metadata_dict)

        # Add batch to Chroma DB collection
        collection.add(
            ids=batch_ids,
            embeddings=batch_embeddings,
            metadatas=batch_metadatas,
            documents=batch_codes
        )


In [5]:
def query_repo(query, collection_name, persist_dir, nvidia_api_key, top_k=5):
    """
    Converts a natural language query into a vector embedding using NVIDIA NIM and searches the
    Chroma vector database to find the top_k most semantically similar code chunks.
    """
    # Connect to NVIDIA NIM API
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=nvidia_api_key
    )
    
    # Call NVIDIA NIM API to convert query to vector representation.
    response = client.embeddings.create(
        input=query,
        model="nvidia/llama-nemotron-embed-1b-v2",
        extra_body={"input_type": "query"}
    )
    query_embedding = response.data[0].embedding

    # Initialize Chroma client
    client_db = chromadb.PersistentClient(path=persist_dir)

    # Retrieve collection
    try:
        collection = client_db.get_collection(name=collection_name)
    except Exception as error:
        print("Error: Collection '", collection_name, "' not found:", error)
        return []

    # Query Chroma using query embedding
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    output = []
    if results is not None:
        if 'documents' in results and results['documents'] is not None:
            if len(results['documents']) > 0:
                docs = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]

                # Map results to our output format
                for idx in range(len(docs)):
                    document_text = docs[idx]
                    metadata = metadatas[idx]
                    distance = distances[idx]
                    
                    # Calculate cosine similarity (1.0 - cosine distance)
                    similarity = 1.0 - distance
                    
                    match_dict = {
                        "code": document_text,
                        "file_path": metadata["file_path"],
                        "class_name": metadata.get("class_name", ""),
                        "name": metadata["name"],
                        "type": metadata.get("type", "function"),
                        "start_line": int(metadata.get("start_line", 0)),
                        "end_line": int(metadata.get("end_line", 0)),
                        "similarity_score": similarity
                    }
                    output.append(match_dict)

    return output


In [6]:
def tokenize_text(text):
    """
    Splits a raw text string into a list of lowercase alphanumeric word tokens.
    """
    words = []
    current_word = []
    for char in text:
        if char.isalnum():
            current_word.append(char.lower())
        else:
            if current_word:
                words.append("".join(current_word))
                current_word = []
    if current_word:
        words.append("".join(current_word))
    return words

def build_bm25_index(chunks):
    """
    Constructs two separate BM25Okapi search indexes over the extracted codebase chunks:
    one for metadata (file path, class, method/function name) and one for the code body.
    """
    metadata_corpus = []
    body_corpus = []
    for chunk in chunks:
        # Build metadata string based on chunk type
        if chunk['type'] == 'method':
            metadata_str = "File: " + chunk['file_path'] + "\nClass: " + chunk['class_name'] + "\nMethod: " + chunk['name']
        elif chunk['type'] == 'class_definition':
            metadata_str = "File: " + chunk['file_path'] + "\nClass: " + chunk['class_name']
        else:
            metadata_str = "File: " + chunk['file_path'] + "\nName: " + chunk['name']
            
        metadata_corpus.append(tokenize_text(metadata_str))
        body_corpus.append(tokenize_text(chunk['code']))
        
    metadata_index = BM25Okapi(metadata_corpus)
    body_index = BM25Okapi(body_corpus)
    return metadata_index, body_index, chunks

def bm25_search(query, metadata_index, body_index, chunks, top_k=10, metadata_weight=3.0, body_weight=1.0):
    """
    Ranks chunks using a field-weighted combination of metadata and code body BM25 scores.
    """
    tokenized_query = tokenize_text(query)
    
    # Get scores from both indexes
    metadata_scores = metadata_index.get_scores(tokenized_query)
    body_scores = body_index.get_scores(tokenized_query)
    
    scored_chunks = []
    for idx in range(len(chunks)):
        m_score = float(metadata_scores[idx])
        b_score = float(body_scores[idx])
        combined_score = (metadata_weight * m_score) + (body_weight * b_score)
        
        scored_chunks.append({
            "chunk": chunks[idx],
            "score": combined_score
        })
        
    scored_chunks.sort(key=lambda x: x["score"], reverse=True)
    
    results = []
    for item in scored_chunks[:top_k]:
        chunk = item["chunk"]
        results.append({
            "code": chunk["code"],
            "file_path": chunk["file_path"],
            "class_name": chunk.get("class_name", ""),
            "name": chunk["name"],
            "type": chunk["type"],
            "start_line": chunk["start_line"],
            "end_line": chunk["end_line"],
            "bm25_score": item["score"]
        })
    return results

def hybrid_search(query, collection_name, persist_dir, metadata_index, body_index, chunks, nvidia_api_key, top_k=5, rrf_k=60, metadata_weight=3.0, body_weight=1.0):
    """
    Fuses dense embedding similarity and field-weighted sparse BM25 keyword matching scores
    using Reciprocal Rank Fusion (RRF).
    """
    # 1. Fetch top 20 candidates from dense search
    dense_results = query_repo(query, collection_name, persist_dir, nvidia_api_key, top_k=20)
    
    # 2. Fetch top 20 candidates from BM25 search
    bm25_results = bm25_search(
        query=query,
        metadata_index=metadata_index,
        body_index=body_index,
        chunks=chunks,
        top_k=20,
        metadata_weight=metadata_weight,
        body_weight=body_weight
    )
    
    # Track rank and scores for both lists
    dense_rank = {}
    dense_scores = {}
    for idx in range(len(dense_results)):
        r = dense_results[idx]
        key = r["file_path"] + "|" + r["name"] + "|" + str(r["start_line"])
        dense_rank[key] = idx + 1
        dense_scores[key] = r["similarity_score"]
        
    bm25_rank = {}
    bm25_scores = {}
    for idx in range(len(bm25_results)):
        r = bm25_results[idx]
        key = r["file_path"] + "|" + r["name"] + "|" + str(r["start_line"])
        bm25_rank[key] = idx + 1
        bm25_scores[key] = r["bm25_score"]
        
    # Combine keys from both lists
    all_keys = set(dense_rank.keys()).union(set(bm25_rank.keys()))
    
    fused_results = []
    for key in all_keys:
        d_rank = dense_rank.get(key)
        rrf_dense = 1.0 / (rrf_k + d_rank) if d_rank is not None else 0.0
            
        b_rank = bm25_rank.get(key)
        rrf_bm25 = 1.0 / (rrf_k + b_rank) if b_rank is not None else 0.0
            
        rrf_score = rrf_dense + rrf_bm25
        
        # Retrieve the original chunk metadata from either of the result lists
        matched_result = None
        for r in dense_results:
            r_key = r["file_path"] + "|" + r["name"] + "|" + str(r["start_line"])
            if r_key == key:
                matched_result = r
                break
        if matched_result is None:
            for r in bm25_results:
                r_key = r["file_path"] + "|" + r["name"] + "|" + str(r["start_line"])
                if r_key == key:
                    matched_result = r
                    break
                    
        if matched_result is not None:
            fused_results.append({
                "code": matched_result["code"],
                "file_path": matched_result["file_path"],
                "class_name": matched_result["class_name"],
                "name": matched_result["name"],
                "type": matched_result.get("type", "function"),
                "start_line": matched_result["start_line"],
                "end_line": matched_result["end_line"],
                "dense_score": dense_scores.get(key, 0.0),
                "bm25_score": bm25_scores.get(key, 0.0),
                "rrf_score": rrf_score
            })
            
    fused_results.sort(key=lambda x: x["rrf_score"], reverse=True)
    return fused_results[:top_k]


In [7]:
import os

# Your GitHub token and repository details
token = os.environ.get("GITHUB_TOKEN", "YOUR_GITHUB_TOKEN")
github_username = "dhruvkachhela"
repo_name = "vibesec"

# Clone the repository directly onto the Kaggle server
print("Cloning private repository...")
exit_code = os.system(f"git clone https://{token}@github.com/dhruvkachhela/vibecheck-scan")

if exit_code == 0:
    print("Success! Repository cloned successfully.")
else:
    print("Error: Failed to clone repository. Please check your username and repository name.")


Cloning private repository...


Cloning into 'vibecheck-scan'...


Success! Repository cloned successfully.


In [8]:
# Generic Kaggle working folder where git clones land
repo_path = "/kaggle/working/vibecheck-scan"
collection_name = "repo_code_chunks"
persist_dir = "./chroma_db"

# Fetch NVIDIA API Key directly inside this cell
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    nvidia_api_key = user_secrets.get_secret("nvidia_api_key")
except Exception:
    nvidia_api_key = "your-nvapi-key-here"

# Automatically clean up old database folder before rebuilding to prevent duplicates
import shutil
shutil.rmtree(persist_dir, ignore_errors=True)

# 1. Extract chunks (Will split classes into methods)
print("Step 1: Chunking codebase inside:", repo_path)
chunks = chunk_repo(repo_path)
print("Successfully extracted", len(chunks), "code chunks.")

# 2. Embed and store using NVIDIA NIM API
if chunks:
    print("\nStep 2: Embedding chunks and storing in Chroma...")
    embed_and_store(chunks, collection_name, persist_dir, nvidia_api_key)
    
    # 3. Build sparse keyword indexes
    print("\nStep 3: Compiling BM25 keyword indexes...")
    metadata_index, body_index, bm25_chunks = build_bm25_index(chunks)
    print("BM25 indexes built successfully.")
    
    print("\nIndexing completed successfully! You can now run Cell 8 below.")


Step 1: Chunking codebase inside: /kaggle/working/vibecheck-scan
Successfully extracted 369 code chunks.

Step 2: Embedding chunks and storing in Chroma...
Initializing NVIDIA NIM API connection...
Connecting to persistent Chroma database at: ./chroma_db
Starting embedding and storage of 369 chunks using NVIDIA NIM...


Embedding and storing chunks: 100%|██████████| 8/8 [00:09<00:00,  1.18s/it]


Step 3: Compiling BM25 keyword indexes...
BM25 indexes built successfully.

Indexing completed successfully! You can now run Cell 8 below.


In [9]:
# Configure same collection details
collection_name = "repo_code_chunks"
persist_dir = "./chroma_db"

# Fetch NVIDIA API Key directly inside this cell
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    nvidia_api_key = user_secrets.get_secret("nvidia_api_key")
except Exception:
    nvidia_api_key = "your-nvapi-key-here"

# List of queries you want to run
queries = [
    "find the orchestrator function",
    "how does the system decide what's a false positive?",
    "which functions call the LLM?"
]

# Loop through each query and print results
for query in queries:
    print("\n" + "=" * 80)
    print(f"HYBRID QUERY: '{query}'")
    print("=" * 80)
    
    # query top_k=2 results, setting metadata_weight=3.0 and body_weight=1.0
    results = hybrid_search(
        query=query,
        collection_name=collection_name,
        persist_dir=persist_dir,
        metadata_index=metadata_index,
        body_index=body_index,
        chunks=bm25_chunks,
        nvidia_api_key=nvidia_api_key,
        top_k=2,
        rrf_k=60,
        metadata_weight=3.0,
        body_weight=1.0
    )
    
    if not results:
        print("No matches retrieved.")
    else:
        for r_idx in range(len(results)):
            result = results[r_idx]
            display_idx = r_idx + 1
            print(f"\nResult {display_idx} (RRF Score: {round(result['rrf_score'], 5)} | Dense Score: {round(result['dense_score'], 4)} | BM25 Score: {round(result['bm25_score'], 4)}):")
            print("File:", result['file_path'])
            if result['class_name']:
                print("Class:", result['class_name'], "| Name:", result['name'])
            else:
                print("Name:", result['name'])
            print("-" * 40)
            
            lines = result['code'].splitlines()
            snippet_lines = []
            for line_idx in range(min(10, len(lines))):
                snippet_lines.append(lines[line_idx])
            snippet = "\n".join(snippet_lines)
            
            if len(lines) > 10:
                snippet += "\n..."
            print(snippet)



HYBRID QUERY: 'find the orchestrator function'

Result 1 (RRF Score: 0.03154 | Dense Score: 0.2722 | BM25 Score: 15.0071):
File: vibesec-pipeline/scanner/orchestrator.py
Name: merge_proximity_findings
----------------------------------------
def merge_proximity_findings(findings: List[Finding], indexer) -> List[Finding]:
    if not findings:
        return []
    # Group findings by (file_path, check_id, check_category, normalized_title, is_false_positive)
    groups = {}
    for f in findings:
        norm_title = (f.title or "").lower().replace("[high]", "").replace("[medium]", "").replace("[critical]", "").replace("[low]", "").strip()
        key = (
            (f.file_path or "").replace("\\", "/").lower(),
            f.check_id or "",
...

Result 2 (RRF Score: 0.03016 | Dense Score: 0.2322 | BM25 Score: 11.6932):
File: vibesec-pipeline/scanner/orchestrator.py
Name: _merge_subgroup
----------------------------------------
def _merge_subgroup(subgroup: List[Finding]) -> Finding:


In [15]:
# How this works:
# This code defines a ReAct (Reasoning + Acting) Agent for codebase QA built using LangGraph.
# 1. AgentState tracks user questions, search history, intermediate thoughts, queries, 
#    final answers, iteration counts, and verifier/critic feedback.
# 2. reasoning_node prompts GLM 5.2 model on NVIDIA NIM to either output an Action (search query) or a Final Answer.
# 3. tool_node executes hybrid search against Chromadb and BM25 index and appends results to history.
# 4. verifier_node (Grounding Critic) evaluates any generated Final Answer strictly against retrieved observations.
# 5. If unsupported claims are found, the graph routes back to reasoning_node with feedback to self-correct.

from typing import List, Tuple, Optional, TypedDict
from openai import OpenAI
from langgraph.graph import StateGraph, END

# Define the State Dictionary for the agent
class AgentState(TypedDict):
    question: str
    history: List[Tuple[str, str, str]]  # list of (thought, action, observation)
    current_thought: str
    action_query: Optional[str]
    final_answer: Optional[str]
    iterations: int
    verification_attempts: int
    verifier_feedback: Optional[str]
    is_grounded: bool


def build_agent_graph(collection_name, persist_dir, metadata_index, body_index, bm25_chunks, nvidia_api_key):
    """
    Creates and compiles a LangGraph StateGraph that defines the ReAct loop with a Grounding Verifier.
    
    Parameters:
        collection_name (str): Name of the Chroma collection used for vector search.
        persist_dir (str): Directory where vector database is persisted.
        metadata_index (dict): Index mapping metadata tokens to code chunk identifiers.
        body_index (dict): Index mapping code body tokens to code chunk identifiers.
        bm25_chunks (list): Raw code chunks used for BM25 keyword matching.
        nvidia_api_key (str): API key for accessing NVIDIA NIM LLM endpoints.
        
    Returns:
        CompiledStateGraph: An executable LangGraph agent pipeline.
    """
    
    def reasoning_node(state: AgentState):
        """
        Prompts GLM model on NVIDIA NIM to reason about the codebase question.
        Generates either an Action (search query) or a Final Answer, considering any past critic feedback.
        
        Parameters:
            state (AgentState): Current dictionary representing the agent state.
            
        Returns:
            dict: Updated state containing current thought, action query, final answer, and iteration count.
        """
        current_iterations = state.get("iterations", 0)
        history_list = state.get("history", [])
        verifier_feedback = state.get("verifier_feedback", None)
        
        history_str = ""
        for idx, (thought, action, observation) in enumerate(history_list):
            history_str += f"\n--- Step {idx+1} ---\n"
            history_str += f"Thought: {thought}\n"
            history_str += f"Action: {action}\n"
            history_str += f"Observation:\n{observation}\n"

        # Inject verifier feedback into the system prompt if previous answer was rejected by critic
        feedback_instruction = ""
        if verifier_feedback:
            feedback_instruction = (
                "\n\nCRITICAL ATTENTION - PREVIOUS ANSWER REJECTED BY VERIFIER:\n"
                f"{verifier_feedback}\n"
                "Your previous answer contained claims unsupported by retrieved code. "
                "You must search for missing details or revise your answer to remove unsupported claims."
            )

        if current_iterations >= 9:
            system_prompt = (
                "You have reached the maximum search limit. You must summarize the best partial answer "
                "using ONLY the information gathered in the observations so far.\n"
                "You must format your output exactly as:\n"
                "Thought: [summarize findings]\n"
                "Final Answer: Incomplete — max iterations reached. [your answer based on available observations]"
                + feedback_instruction
            )
        else:
            system_prompt = (
                "You are an expert codebase QA agent. You have access to a hybrid search tool to locate "
                "relevant code chunks (functions and classes) from the codebase.\n\n"
                "Your task is to answer the user's question about the codebase.\n"
                "For each turn, you can either perform a search or provide a final answer.\n\n"
                "Format requirements:\n"
                "- If you need to search the codebase, output exactly:\n"
                "Thought: [describe what you need to look up and why]\n"
                "Action: [your query string for search]\n\n"
                "- If you have gathered enough information to answer the question, output exactly:\n"
                "Thought: [summarize your findings]\n"
                "Final Answer: [your complete answer based on the retrieved code chunks]\n\n"
                "Do not include any other text outside these fields."
                + feedback_instruction
            )
        
        user_prompt = f"Original Question: {state['question']}\n\nSearch History:\n{history_str}\n\nPlease take the next step."
        
        # Instantiate OpenAI client pointing to NVIDIA NIM API endpoint
        client = OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=nvidia_api_key
        )
        
        # Request completion from NVIDIA NIM LLM
        response = client.chat.completions.create(
            model="z-ai/glm-5.2",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.1,
            max_tokens=16384
        )
        
        response_text = response.choices[0].message.content.strip()
        
        thought = ""
        action_query = None
        final_answer = None
        
        # Parse output fields explicitly
        if "Thought:" in response_text:
            parts = response_text.split("Thought:", 1)
            rest = parts[1]
            if "Action:" in rest:
                thought_part, action_part = rest.split("Action:", 1)
                thought = thought_part.strip()
                action_query = action_part.strip()
            elif "Final Answer:" in rest:
                thought_part, answer_part = rest.split("Final Answer:", 1)
                thought = thought_part.strip()
                final_answer = answer_part.strip()
            else:
                thought = rest.strip()
        else:
            if "Final Answer:" in response_text:
                final_answer = response_text.split("Final Answer:", 1)[1].strip()
            elif "Action:" in response_text:
                action_query = response_text.split("Action:", 1)[1].strip()
            else:
                final_answer = response_text

        print(f"\n--- [AGENT REASONING] Iteration {current_iterations + 1} ---")
        print(f"Thought: {thought}")
        if final_answer:
            print(f"Final Answer Generated (Pending Verification): {final_answer}")
        else:
            print(f"Action/Query: {action_query}")
            
        return {
            "current_thought": thought,
            "action_query": action_query,
            "final_answer": final_answer,
            "iterations": current_iterations + 1
        }

    def tool_node(state: AgentState):
        """
        Executes the hybrid search tool using the query generated by the reasoning node.
        Appends the resulting code block matches as an observation in the history.
        
        Parameters:
            state (AgentState): Current state containing action query string.
            
        Returns:
            dict: Updated state containing new history entry and reset action query.
        """
        action = state["action_query"]
        
        results = hybrid_search(
            query=action,
            collection_name=collection_name,
            persist_dir=persist_dir,
            metadata_index=metadata_index,
            body_index=body_index,
            chunks=bm25_chunks,
            nvidia_api_key=nvidia_api_key,
            top_k=5,
            metadata_weight=3.0,
            body_weight=1.0
        )
        
        # Format observations into readable context
        observation = ""
        for r in results:
            observation += f"File: {r['file_path']} | Name: {r['name']} | Type: {r['type']}\n"
            observation += f"Code:\n{r['code']}\n"
            observation += "-" * 40 + "\n"
            
        if not results:
            observation = "No matching code chunks found."
            
        new_history = list(state["history"])
        new_history.append((state["current_thought"], action, observation))
        
        print(f"\n--- [TOOL CALL] ---")
        print(f"Query: '{action}' -> Retrieved {len(results)} chunks.")
        
        return {
            "history": new_history,
            "action_query": None
        }

    def verifier_node(state: AgentState):
        """
        Runs a secondary critic LLM call to verify if every claim in the final answer
        is supported by the retrieved code observations.
        
        Parameters:
            state (AgentState): Current agent state containing final answer and history observations.
            
        Returns:
            dict: State updates with grounding status, verifier feedback, and attempt count.
        """
        final_answer = state["final_answer"]
        history_list = state.get("history", [])
        attempts = state.get("verification_attempts", 0) + 1
        
        # Consolidate all observations from search history into a single block
        all_observations = ""
        for step_idx, (thought, action, observation) in enumerate(history_list):
            all_observations += f"\n--- Observation from Step {step_idx + 1} (Query: {action}) ---\n"
            all_observations += observation + "\n"
            
        if not all_observations.strip():
            all_observations = "No code chunks were retrieved during search."
            
                verifier_system_prompt = (
            "You are a strict Grounding Critic for a codebase QA system.\n"
            "Your task is to evaluate every claim made in the proposed Final Answer against the provided retrieved code observations.\n\n"
            "You MUST evaluate each claim against these three categories:\n"
            "1. SUPPORTED — PARAPHRASE: The claim restates something in a single observation using different wording, synonyms, or naming variations "
            "(e.g., 'OpenAI(...)' instantiation supports 'imports/uses the OpenAI client'; 'NVIDIA NIM 70B' supports '70B parameter model').\n"
            "2. SUPPORTED — SYNTHESIS: The claim is a valid conclusion derived by combining information across TWO OR MORE observations, "
            "even if no single observation states it outright (e.g., combining function definitions and call hierarchy observations to conclude a function is the entry point).\n"
            "3. UNSUPPORTED — FABRICATION: The claim states a specific fact (a number, function/class name, file path, or call relationship) "
            "that has ZERO textual or inferential basis anywhere across the retrieved observations.\n\n"
            "Critical Instructions:\n"
            "- You MUST explicitly attempt synthesis across all observations before marking any claim UNSUPPORTED. Check combinations of observations, not just individual snippets.\n"
            "- Treat close paraphrases, naming variations, and code equivalents as SUPPORTED, not UNSUPPORTED.\n"
            "- Reserve UNSUPPORTED strictly for true fabrication with no supporting evidence or logical synthesis path in the observations.\n\n"
            "Output Requirements:\n"
            "- If all claims are either PARAPHRASE or SYNTHESIS, respond with:\n"
            "VERDICT: SUPPORTED\n\n"
            "- If any claim is FABRICATION, respond with:\n"
            "VERDICT: UNSUPPORTED\n"
            "Unsupported Claims:\n"
            "- [UNSUPPORTED — FABRICATION]: [describe the fabricated claim and why no snippet or combination supports it]\n\n"
            "Feedback Instructions for Reasoning Node:\n"
            "- [Provide explicit, actionable instructions telling the agent what missing code/file to search for or which unverified claims to remove]\n"
        )
        
        verifier_user_prompt = (
            f"Question: {state['question']}\n\n"
            f"Retrieved Code Observations:\n{all_observations}\n\n"
            f"Proposed Final Answer:\n{final_answer}\n\n"
            "Evaluate grounding now."
        )
        
        # Connect to NVIDIA NIM API client for verifier LLM call
        client = OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=nvidia_api_key
        )
        
        # Send critique request to model
        response = client.chat.completions.create(
            model="z-ai/glm-5.2",
            messages=[
                {"role": "system", "content": verifier_system_prompt},
                {"role": "user", "content": verifier_user_prompt}
            ],
            temperature=0.0,
            max_tokens=4096
        )
        
        verifier_text = response.choices[0].message.content.strip()
        print(f"\n--- [GROUNDING VERIFIER CRITIC] Attempt {attempts} ---")
        print(verifier_text)
        
        # Robust verdict check (prevents false positives while handling typos like SUPPORTD)
        upper_text = verifier_text.upper()
        is_grounded = "VERDICT: SUPPORT" in upper_text and "VERDICT: UNSUPPORT" not in upper_text

        
        if is_grounded or attempts >= 2:
            updated_answer = final_answer
            if not is_grounded:
                updated_answer += f"\n\n[WARNING: Partially Grounded]\nVerifier Feedback:\n{verifier_text}"
            return {
                "is_grounded": is_grounded,
                "verifier_feedback": verifier_text,
                "verification_attempts": attempts,
                "final_answer": updated_answer
            }
        else:
            return {
                "is_grounded": False,
                "verifier_feedback": verifier_text,
                "verification_attempts": attempts,
                "final_answer": None
            }

    # Define Graph workflow
    workflow = StateGraph(AgentState)
    workflow.add_node("reasoning", reasoning_node)
    workflow.add_node("tool", tool_node)
    workflow.add_node("verifier", verifier_node)
    
    workflow.set_entry_point("reasoning")

    # Routing logic after reasoning node
    def route_agent(state: AgentState):
        if state["final_answer"] is not None:
            return "verifier"
        if state["iterations"] >= 10:
            return "verifier"
        return "tool"

    # Routing logic after verifier node
    def route_verification(state: AgentState):
        if state["final_answer"] is not None:
            return "end"
        return "re_reason"

    workflow.add_conditional_edges(
        "reasoning",
        route_agent,
        {
            "verifier": "verifier",
            "tool": "tool"
        }
    )
    
    workflow.add_conditional_edges(
        "verifier",
        route_verification,
        {
            "end": END,
            "re_reason": "reasoning"
        }
    )

    workflow.add_edge("tool", "reasoning")
    
    return workflow.compile()


In [16]:
# 1. Compile the LangGraph agent graph
print("Compiling ReAct Agent Graph with Grounding Verifier...")
agent_app = build_agent_graph(
    collection_name=collection_name,
    persist_dir=persist_dir,
    metadata_index=metadata_index,
    body_index=body_index,
    bm25_chunks=bm25_chunks,
    nvidia_api_key=nvidia_api_key
)
print("Agent graph compiled successfully.")

# 2. Define test questions
agent_questions = [
    "what LLM and it's framework are we using in it?",
    "what function is the pipeline's entry point?"
]

# 3. Run the questions and output traces
print("\nTesting ReAct Agent Loop with Critic:")
for question in agent_questions:
    print("\n" + "=" * 90)
    print(f"QUESTION: '{question}'")
    print("=" * 90)
    
    initial_state = {
        "question": question,
        "history": [],
        "current_thought": "",
        "action_query": None,
        "final_answer": None,
        "iterations": 0,
        "verification_attempts": 0,
        "verifier_feedback": None,
        "is_grounded": False
    }
    
    final_state = agent_app.invoke(initial_state)
    
    print("\n" + "-" * 90)
    print("FINAL VERIFIED AGENT ANSWER:")
    print(final_state["final_answer"])
    print("-" * 90)


Compiling ReAct Agent Graph with Grounding Verifier...
Agent graph compiled successfully.

Testing ReAct Agent Loop with Critic:

QUESTION: 'what LLM and it's framework are we using in it?'

--- [AGENT REASONING] Iteration 1 ---
Thought: I need to search the codebase to find what LLM and framework are being used. I'll look for common LLM-related imports, configurations, or dependencies.
Action/Query: LLM framework openai langchain anthropic model configuration

--- [TOOL CALL] ---
Query: 'LLM framework openai langchain anthropic model configuration' -> Retrieved 5 chunks.

--- [AGENT REASONING] Iteration 2 ---
Thought: The search results show LLM vulnerability scanning code but don't reveal what LLM/framework the project itself uses. I need to look at configuration files, dependencies, or main application code to find the actual LLM integration.
Action/Query: package.json requirements.txt pyproject.toml openai anthropic langchain llm configuration

--- [TOOL CALL] ---
Query: 'package.j